# Courbes de résultats — Grand Tunis v4 et prévision v5

Colab **CPU**. Le carnet ne charge aucun modèle et ne relit pas le test : il trace les résultats déjà écrits sur le Drive (`history.json`, `test_par_niveau_synoptique.csv`, `prevision_bilan.json`). Les figures sont aussi copiées dans `MyDrive/pcfno3d_courbes/`. La session est libérée à la fin.

In [ ]:
from google.colab import drive
import os
if not os.path.isdir("/content/drive/MyDrive"):
    drive.mount("/content/drive")
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

RUNS = Path("/content/drive/MyDrive/pcfno3d_grand_tunis/runs")
OS7 = Path("/content/drive/MyDrive/pcfno3d_grand_tunis/os7")
FIG = Path("/content/drive/MyDrive/pcfno3d_courbes")          # dossier neuf : rien n'est ecrase ailleurs
FIG.mkdir(exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

def historique(run):
    h = json.loads((RUNS / run / "history.json").read_text())
    h = [r for r in h if r.get("epoch", -1) >= 0 and isinstance(r.get("train"), dict)]
    df = pd.DataFrame({"epoque": [r["epoch"] + 1 for r in h],
                       "train": [r["train"].get("data") for r in h],
                       "validation": [r["val"].get("data") for r in h],
                       "r2_u": [r.get("val_r2_u") for r in h], "r2_v": [r.get("val_r2_v") for r in h],
                       "r2_w": [r.get("val_r2_w") for r in h]})
    print(run, ":", len(df), "epoques lues dans", RUNS / run / "history.json")
    return df

def courbes_entrainement(df, titre, nom):
    fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
    ax[0].plot(df["epoque"], df["train"], "o-", label="entrainement")
    ax[0].plot(df["epoque"], df["validation"], "s-", label="validation")
    ax[0].set_yscale("log"); ax[0].set_xlabel("Epoque"); ax[0].set_title("Erreur sur les donnees (normalisee)"); ax[0].legend()
    for c, m in (("r2_u", "o"), ("r2_v", "s"), ("r2_w", "^")):
        ax[1].plot(df["epoque"], df[c], m + "-", label="R2 " + c[-1])
    ax[1].axhline(0.95, color="red", ls="--", label="seuil 0,95")
    ax[1].set_xlabel("Epoque"); ax[1].set_title("R2 sur la validation"); ax[1].legend()
    fig.suptitle(titre); fig.tight_layout(); fig.savefig(FIG / nom, dpi=150); plt.show()


## 1. Grand Tunis v4 : entraînement et validation

In [ ]:
# 1. Grand Tunis v4 : entrainement et validation (affinage de 15 epoques)
v4 = historique("grand_tunis_v4_dso")
courbes_entrainement(v4, "Grand Tunis v4 — entrainement et validation (history.json)", "01_gt_v4_entrainement.png")
print(v4.round(4).to_string(index=False))


## 2. Grand Tunis : toute la chaîne d'entraînement

In [ ]:
# 2. Grand Tunis : toute la chaine v2 (20 epoques) -> v2_long (+40) -> v4 (+15), validation
chaine, debut = [], 0
for run in ("grand_tunis_v2", "grand_tunis_v2_long", "grand_tunis_v4_dso"):
    try:
        d = historique(run); d["epoque_totale"] = d["epoque"] + debut; d["run"] = run
        chaine.append(d); debut = int(d["epoque_totale"].max())
    except FileNotFoundError:
        print(run, ": history.json absent, saute")
if chaine:
    c = pd.concat(chaine)
    fig, ax = plt.subplots(figsize=(13, 4.2))
    ax.plot(c["epoque_totale"], c["train"], label="entrainement")
    ax.plot(c["epoque_totale"], c["validation"], label="validation")
    for d in chaine[1:]:
        ax.axvline(d["epoque_totale"].min() - 0.5, color="grey", ls=":")
    ax.set_yscale("log"); ax.set_xlabel("Epoque cumulee (v2 | v2_long | v4)"); ax.legend()
    ax.set_title("Grand Tunis — erreur d'entrainement et de validation sur toute la chaine")
    fig.tight_layout(); fig.savefig(FIG / "02_gt_chaine_entrainement.png", dpi=150); plt.show()


## 3. Grand Tunis v4 : test 2022 par altitude

In [ ]:
# 3. Grand Tunis v4 : resultats du TEST 2022 par altitude (fichier deja ecrit le 24/10, le test n'est pas relu)
f = RUNS / "grand_tunis_v4_dso" / "eval_pcfno3d" / "best_wcont" / "test_par_niveau_synoptique.csv"
t = pd.read_csv(f); print("lu :", f); print(t.round(3).to_string(index=False))
t = t[t["couverture"] >= 0.05]
fig, ax = plt.subplots(1, 2, figsize=(13, 4.6))
for c, m in (("u", "o"), ("v", "s"), ("w", "^")):
    ax[0].plot(t[f"r2_{c}"], t["z_m"], m + "-", label=c)
    ax[1].plot(t[f"rmse_{c}"], t["z_m"], m + "-", label=c)
ax[0].axvline(0.95, color="red", ls="--", label="seuil 0,95"); ax[1].axvline(0.5, color="red", ls="--", label="seuil 0,5 m/s")
ax[0].set_xlabel("R2"); ax[1].set_xlabel("RMSE (m/s)")
for a in ax: a.set_ylabel("Altitude ASL (m)"); a.legend()
ax[0].set_title("R2 par niveau (test 2022)"); ax[1].set_title("RMSE par niveau (test 2022)")
fig.suptitle("Grand Tunis v4 — test 2022, 180 instants (w par continuite, modele servi)")
fig.tight_layout(); fig.savefig(FIG / "03_gt_v4_test_niveaux.png", dpi=150); plt.show()


## 4. Prévision v5 : entraînement et validation

In [ ]:
# 4. Prevision v5 : affinage de 12 epoques sur la prevision GFS a +24 h
v5 = historique("grand_tunis_v5_gfs")
courbes_entrainement(v5, "Prevision v5 — entrainement et validation (history.json)", "04_v5_entrainement.png")
print(v5.round(4).to_string(index=False))


## 5. Prévision v5 : stations réelles, +24 h et +48 h

In [ ]:
# 5. Prevision v5 : erreur aux stations reelles en 2022, ATOM Wind contre GFS brut et persistance
import numpy as np
fig, ax = plt.subplots(1, 2, figsize=(13, 4.4), sharey=True)
for a, e in zip(ax, (24, 48)):
    f = OS7 / f"v5_f{e}" / "prevision_bilan.json"
    v = json.loads(f.read_text())["verdicts"]; print("lu :", f)
    noms = [k for k in v if k != "toutes"] + ["toutes"]
    x = np.arange(len(noms)); l = 0.27
    for k, (s, lib) in enumerate((("prevision", "ATOM Wind v5"), ("gfs", "GFS brut"), ("persistance", "persistance"))):
        val = [v[n]["rmse_vect"][s] for n in noms]
        b = a.bar(x + (k - 1) * l, val, l, label=lib)
        a.bar_label(b, fmt="%.2f", fontsize=7)
    a.set_xticks(x); a.set_xticklabels(noms, rotation=15); a.set_title(f"Echeance +{e} h")
    a.set_ylabel("Erreur vectorielle a 10 m (m/s)")
ax[0].legend(); fig.suptitle("Prevision v5 — test 2022 aux stations reelles (prevision_bilan.json)")
fig.tight_layout(); fig.savefig(FIG / "05_v5_prevision_stations.png", dpi=150); plt.show()
print("figures ->", FIG, sorted(p.name for p in FIG.iterdir()))


In [ ]:
# fin : la session CPU est liberee (les courbes restent affichees dans le carnet et copiees sur le Drive)
from google.colab import runtime
runtime.unassign()
